In [0]:
import logging
from pyspark.sql.functions import current_timestamp, col, regexp_extract

# ==============================================================================
# PASO 1: CONFIGURACIÓN GLOBAL (Adiós a la tabla de control)
# ==============================================================================
dbutils.widgets.text("catalog_param", "")
dbutils.widgets.text("source_schema_param", "")
dbutils.widgets.text("target_schema_param", "")
# 2. Capturar los valores inyectados por el YAML
catalog_name = dbutils.widgets.get("catalog_param")
source_schema = dbutils.widgets.get("source_schema_param")
target_schema = dbutils.widgets.get("target_schema_param")


spark.catalog.setCurrentCatalog(catalog_name)

# Apuntamos a la RAÍZ. Un solo vigía para todos los portales.
base_landing_path = f"/Volumes/{catalog_name}/{source_schema}/real_state_csv_landing_volume/"
checkpoint_path = f"{base_landing_path}_checkpoints/inmuebles_global_v3/"
schema_location_path = f"{base_landing_path}_schemas/inmuebles_global_v3/"
#checkpoint_path = f"{base_landing_path}_checkpoints/inmuebles_global/"
#schema_location_path = f"{base_landing_path}_schemas/inmuebles_global/"
target_table = f"{target_schema}.inmuebles"

print("Iniciando escudo de ingesta global optimizado...")

try:
    # ==========================================================================
    # PASO 2: LECTURA GLOBAL (El Escudo Nativo)
    # ==========================================================================
    # Auto Loader verifica la cola SQS de AWS. Si no hay nada, el trigger(availableNow) 
    # terminará la ejecución limpiamente sin iterar sobre archivos.
    df_raw = spark.readStream \
        .format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("header", "true") \
        .option("multiLine", "true") \
        .option("escape", "\"") \
        .option("cloudFiles.rescuedDataColumn", "_rescued_data") \
        .option("cloudFiles.schemaLocation", schema_location_path) \
        .option("cloudFiles.useNotifications", "false") \
        .load(base_landing_path)

    # ==========================================================================
    # PASO 3: ENRIQUECIMIENTO DINÁMICO (Regex Magic)
    # ==========================================================================
    # Extraemos scraper, portal, tipo y operación directamente del nombre del CSV
    # Ejemplo: .../Selenium1_inmuebles24_departamento_venta_2026-05-31.csv
    
    # Patrón Regex para capturar los 5 elementos separados por guión bajo
    regex_pattern = r"/([^/]+)_([^_]+)_([^_]+)_([^_]+)_(\d{4}-\d{2}-\d{2})\.csv$"

    df_enriched = df_raw \
        .withColumn("source_file", col("_metadata.file_path")) \
        .withColumn("scraper_tool", regexp_extract("source_file", regex_pattern, 1)) \
        .withColumn("portal", regexp_extract("source_file", regex_pattern, 2)) \
        .withColumn("property_type", regexp_extract("source_file", regex_pattern, 3)) \
        .withColumn("operation_type", regexp_extract("source_file", regex_pattern, 4)) \
        .withColumn("extraction_date", regexp_extract("source_file", regex_pattern, 5)) \
        .withColumn("ingested_at", current_timestamp())

    # ==========================================================================
    # PASO 4: ESCRITURA DIRECTA
    # ==========================================================================
    query = df_enriched.writeStream \
        .format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", checkpoint_path) \
        .option("mergeSchema", "true") \
        .trigger(availableNow=True) \
        .toTable(target_table)

    query.awaitTermination()
    
    # ==========================================================================
    # PASO 5: REPORTE DEL ESCUDO
    # ==========================================================================
    metrics = query.lastProgress
    num_rows = metrics.get('numInputRows', 0) if metrics else 0
    
    if num_rows == 0:
        print("🛡️ ESCUDO ACTIVO: No se detectaron archivos nuevos en la cola. Clúster liberado sin gastar cómputo.")
    else:
        print(f"✅ ÉXITO: Pipeline finalizado. Se ingirieron y etiquetaron {num_rows} filas nuevas hacia {target_table}.")

except Exception as e:
    error_str = str(e)
    if "CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE" in error_str:
        print("⚠️ AVISO: El volumen de aterrizaje está completamente vacío. Sube tus CSV para que Auto Loader genere el esquema.")
    else:
        print(f"💥 ERROR FATAL en la ingesta global: {error_str}")